# Plant Disease VQA — Google Colab training

Run cells manually in order. No benchmark launches automatically. Project code and temporary images live under /content; only durable artifacts are copied to Drive. Runtime setup and training cells require a Colab GPU.

## 1. Choose the project source

Set a public GitHub URL or upload a project ZIP/folder through the Files panel. Pin PROJECT_REF to a commit for reproducible runs. Do not clone into Drive.

In [ ]:
from pathlib import Path
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"  # PyTorch-only Transformers backend

PROJECT_URL = "https://github.com/dinhdat081026/PlantDisease_VQA.git"  # replace
PROJECT_REF = "main"  # preferably pin a commit SHA
UPLOAD_ZIP = ""  # optional: /content/plant-vqa-project.zip
PROJECT_DIR = Path("/content/plant_disease_visual_question_answering")
DATA_DIR = Path("/content/data/PlantVillageVQA")
OUTPUT_DIR = Path("/content/outputs")

if UPLOAD_ZIP:
    import zipfile
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(UPLOAD_ZIP) as zf:
        root = PROJECT_DIR.resolve()
        for member in zf.infolist():
            target = (PROJECT_DIR / member.filename).resolve()
            if not target.is_relative_to(root) or ".." in Path(member.filename).parts:
                raise ValueError(f"Unsafe project archive member: {member.filename}")
        zf.extractall(PROJECT_DIR)
elif not (PROJECT_DIR / "src/train.py").exists():
    if "OWNER" in PROJECT_URL:
        raise ValueError("Set PROJECT_URL to your GitHub repository or upload the project ZIP")
    !git clone --branch {PROJECT_REF} --depth 1 {PROJECT_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print("Project:", PROJECT_DIR)
print("Data:", DATA_DIR)
print("Outputs:", OUTPUT_DIR)

: 

## 2. Verify accelerator and install dependencies

This project is a source tree with a `src/` package, but no pyproject.toml/setup.py and no installed console entry points. Editable installation is unnecessary: run `python -m src.train` from PROJECT_DIR and add PROJECT_DIR to sys.path for notebook imports. The pip command deliberately shows full diagnostics. It never reinstalls torch or torchvision.

In [ ]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
import subprocess, sys, importlib.util

for module in ("torch", "torchvision"):
    if importlib.util.find_spec(module) is None:
        raise RuntimeError(f"Colab runtime is missing {module}; select a CUDA-enabled runtime")
import torch
print("Python:", sys.version)
# Python 3.13 is acceptable only when Colab’s preinstalled torch/torchvision
# are a mutually compatible CUDA build. The checked-in 2.6.0/0.21.0 pins
# have CPython 3.13 wheels, but this cell intentionally does not install them.
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime (Runtime > Change runtime type > T4 GPU)")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

# Keep the active CUDA PyTorch/torchvision pair untouched.
requirements = (PROJECT_DIR / "requirements.txt").read_text().splitlines()
install = [line.strip() for line in requirements if line.strip() and not line.lstrip().startswith("#") and line.split("==")[0].strip().lower() not in {"torch", "torchvision"}]
subprocess.check_call([sys.executable, "-m", "pip", "install", *install])

# Source-tree import: no editable install and no packaging metadata required.
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
import src.config, src.data, src.models, src.train, src.evaluate
print("Project modules imported from:", PROJECT_DIR)
print("No editable install is required.")

## 3. Mount Drive for durable storage

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/PlantDiseaseVQA")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Durable outputs:", DRIVE_ROOT)

## 4. Download or locate dataset

Dataset is stored temporarily under /content. Download uses the repository CLI and Hugging Face revision discovery. To use an uploaded archive, extract it into DATA_DIR and verify it has one CSV with the six source columns. Never put the dataset in Drive for training.

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
source_csv = sorted(DATA_DIR.rglob("*.csv"))
if not source_csv:
    subprocess.check_call([sys.executable, "-m", "src.download", "--data_dir", str(DATA_DIR)], cwd=PROJECT_DIR)
source_csv = list(DATA_DIR.rglob("*.csv"))
if len(source_csv) != 1:
    raise RuntimeError(f"Expected exactly one discovered annotation CSV, got {source_csv}; set an explicit --csv in later cells")
SOURCE_CSV = source_csv[0]
print("Annotation:", SOURCE_CSV)
print("Image files:", sum(1 for p in DATA_DIR.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}))

## 5. EDA, tests and a model smoke test

Run EDA and tests before training. Then construct a fresh-initialized batch-1 ViT+BERT model and run one inference forward to check actual imports, tensor wiring and CUDA memory. This does not train or save a checkpoint.

In [ ]:
subprocess.check_call([sys.executable, "-m", "src.eda", "--data_dir", str(DATA_DIR), "--csv", str(SOURCE_CSV), "--output_dir", str(OUTPUT_DIR)], cwd=PROJECT_DIR)
subprocess.check_call([sys.executable, "-m", "src.verify_numbers", "--data_dir", str(DATA_DIR), "--csv", str(SOURCE_CSV), "--output", str(OUTPUT_DIR / "results/verification.json")], cwd=PROJECT_DIR)

# Make real-data tests inspect the CSV downloaded in this notebook instead of
# silently skipping because it is outside the repository's default data path.
test_env = os.environ.copy()
test_env["PLANTVQA_CSV"] = str(SOURCE_CSV)
test_command = [sys.executable, "-m", "pytest", "tests/test_data.py", "tests/test_dataset_pipeline.py", "tests/test_models.py", "tests/test_train_evaluate.py", "-v"]
print("Running:", " ".join(test_command))
test_result = subprocess.run(test_command, cwd=PROJECT_DIR, env=test_env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(test_result.stdout, end="" if test_result.stdout.endswith("\n") else "\n")
if test_result.returncode:
    raise RuntimeError(f"pytest failed with exit code {test_result.returncode}; see the complete pytest failure output above.")

# Tiny random-weight forward smoke test; no optimizer, gradients or training.
import torch
from src.models import ViTBERT
smoke_model = ViTBERT(num_classes=2, mode="full", pretrained=False).to("cuda").eval()
smoke_batch = {
    "pixel_values": torch.zeros(1, 3, 224, 224, device="cuda"),
    "input_ids": torch.tensor([[101, 102]], dtype=torch.long, device="cuda"),
    "attention_mask": torch.ones(1, 2, dtype=torch.long, device="cuda"),
    "token_type_ids": torch.zeros(1, 2, dtype=torch.long, device="cuda"),
}
with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
    smoke_logits = smoke_model(**smoke_batch)
assert smoke_logits.shape == (1, 2) and torch.isfinite(smoke_logits).all()
print("CUDA ViTBERT smoke test OK:", smoke_logits.shape, smoke_logits.dtype)
del smoke_model, smoke_batch, smoke_logits
torch.cuda.empty_cache()

## 6. Set the Colab resource profile

The project base batch size is 16 with accumulation 1. This notebook uses microbatch 4 with accumulation 4 (effective batch size 16) to reduce T4 activation memory. This is an explicit runtime override; it changes microbatch execution and dropout sampling. If VRAM is still insufficient, use batch 2/accumulation 8, then batch 1/accumulation 16.

In [ ]:
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 4
NUM_WORKERS = 2

# Explicit override only: keep the original official test whenever possible.
TEST_POLICY = "subset"  # mini/class total budgets are smaller than all factoid test images

assert BATCH_SIZE * GRADIENT_ACCUMULATION == 16
print({"microbatch": BATCH_SIZE, "gradient_accumulation_steps": GRADIENT_ACCUMULATION,
       "effective_batch_size": BATCH_SIZE * GRADIENT_ACCUMULATION, "test_policy": TEST_POLICY})

## 7. Validate data for each planned run

Run this cell before each configuration. Validation runs selected-image decoding and pytest without skipped tests; no model is initialized in validation-only mode. It can take several minutes for full data.

In [ ]:
def validate_run(level, mode):
    report = OUTPUT_DIR / "results" / f"data_validation_{level}_subset.json"
    cmd = [sys.executable, "-m", "src.train", "--level", level, "--mode", mode,
           "--data_dir", str(DATA_DIR), "--csv", str(SOURCE_CSV),
           "--output_dir", str(OUTPUT_DIR), "--test-policy", TEST_POLICY,
           "--batch-size", str(BATCH_SIZE), "--gradient-accumulation-steps", str(GRADIENT_ACCUMULATION),
           "--num-workers", str(NUM_WORKERS), "--validate-only", "--validation-report", str(report)]
    subprocess.check_call(cmd, cwd=PROJECT_DIR)
    return report

print("Validation helper ready. No run has started.")

## 8. Train mini, class, ablations and full

Each code cell is an explicit user action. Cell outputs are saved locally first and copied to Drive after each run. This sequence is intentionally not executed on notebook open.

In [ ]:
def train_run(level, mode="full", *, resume=False):
    report = validate_run(level, mode)
    cmd = [sys.executable, "-m", "src.train", "--level", level, "--mode", mode,
           "--data_dir", str(DATA_DIR), "--csv", str(SOURCE_CSV),
           "--output_dir", str(OUTPUT_DIR), "--test-policy", TEST_POLICY,
           "--batch-size", str(BATCH_SIZE), "--gradient-accumulation-steps", str(GRADIENT_ACCUMULATION),
           "--num-workers", str(NUM_WORKERS), "--validation-report", str(report)]
    if resume:
        cmd.append("--resume")
    subprocess.check_call(cmd, cwd=PROJECT_DIR)
    backup_outputs()

def backup_outputs():
    import shutil
    for folder in ("models", "results", "figures"):
        source = OUTPUT_DIR / folder
        if source.exists():
            destination = DRIVE_ROOT / folder
            destination.mkdir(parents=True, exist_ok=True)
            shutil.copytree(source, destination, dirs_exist_ok=True)

print("Training helper ready. Call one run explicitly, for example train_run('mini', 'full').")

In [ ]:
if False:  # Set to True only when intentionally launching this configuration
    train_run("mini", "full")

In [ ]:
if False:  # Set to True only when intentionally launching this configuration
    train_run("class", "full")

In [ ]:
if False:  # Set to True only when intentionally launching this configuration
    train_run("class", "frozen")

In [ ]:
if False:  # Set to True only when intentionally launching this configuration
    train_run("class", "question_only")

In [ ]:
if False:  # Set to True only when intentionally launching this configuration
    train_run("class", "image_only")

In [ ]:
if False:  # Set to True only when intentionally launching this configuration
    train_run("full", "full")

## 9. Evaluate a selected checkpoint

Choose a concrete checkpoint path. Evaluation checks source, selected images, split and preprocessing fingerprints before inference. It evaluates the configured test view; subset policy is not the complete official test.

In [ ]:
# Set this to a specific best checkpoint if desired; otherwise use the newest best checkpoint found.
CHECKPOINT = None  # e.g. OUTPUT_DIR / "models/full_class.pt"

if CHECKPOINT is None:
    checkpoint_dirs = [OUTPUT_DIR / "models", DRIVE_ROOT / "models"]
    available = []
    for checkpoint_dir in checkpoint_dirs:
        if checkpoint_dir.is_dir():
            available.extend(p for p in checkpoint_dir.glob("*.pt") if not p.stem.endswith("_last"))
    if not available:
        details = "\n".join(f"  {directory}: " + (", ".join(p.name for p in directory.glob("*.pt")) if directory.is_dir() else "directory not found") for directory in checkpoint_dirs)
        raise FileNotFoundError(
            "No best model checkpoint was found. Run a training configuration first, "
            "or set CHECKPOINT to its .pt file. Searched:\n" + details)
    CHECKPOINT = max(available, key=lambda path: path.stat().st_mtime)
else:
    CHECKPOINT = Path(CHECKPOINT)
    if not CHECKPOINT.is_absolute():
        CHECKPOINT = PROJECT_DIR / CHECKPOINT

if not CHECKPOINT.is_file():
    raise FileNotFoundError(f"Checkpoint does not exist: {CHECKPOINT}")
print("Evaluating checkpoint:", CHECKPOINT)
subprocess.check_call([sys.executable, "-m", "src.evaluate", "--ckpt", str(CHECKPOINT),
    "--data_dir", str(DATA_DIR), "--csv", str(SOURCE_CSV), "--output_dir", str(OUTPUT_DIR / "results")], cwd=PROJECT_DIR)
backup_outputs()

## 10. Copy durable results to Drive

In [ ]:
backup_outputs()
print("Drive backup complete:", DRIVE_ROOT)
print("Checkpoints:", sorted(p.name for p in (DRIVE_ROOT / "models").glob("*.pt")))
print("Results:", len(list((DRIVE_ROOT / "results").glob("*"))))